In [1]:
import math
import pandas as pd
from scipy.stats import norm

# =========================
# ΔΕΔΟΜΕΝΑ ΑΠΟ ΤΗΝ ΕΚΦΩΝΗΣΗ
# =========================

# SKU A (EOQ)
DA = 48_000
KA = 240
hA = 1.20

# SKU B ((Q,R))
muB = 220        # μέση τιμή ημερήσιας ζήτησης
sigmaB = 60      # τυπική απόκλιση ημερήσιας ζήτησης
L = 4            # lead time (ημέρες)
DB_annual = 80_000
KB = 180
hB = 2.50
CSL = 0.95       # Cycle Service Level

# SKU C (Newsvendor)
cC = 18
pC = 39
vC = 6
muC = 4800       # μέση ζήτηση περιόδου
sigmaC = 1400    # τυπική απόκλιση ζήτησης περιόδου

# =========================
# SKU A – EOQ
# =========================

EOQ_A = math.sqrt(2 * DA * KA / hA)                 # Q* = sqrt(2DK/h)
orders_A = DA / EOQ_A                               # παραγγελίες/έτος = D/Q
cost_A = (DA / EOQ_A) * KA + (EOQ_A / 2) * hA       # ordering + holding

results_A = pd.DataFrame([{
    "EOQ_A": EOQ_A,
    "Orders_per_year_A": orders_A,
    "Annual_cost_A(ordering+holding)": cost_A
}])

# Sensitivity ±20% σε DA και hA
levels = [-0.2, 0.2]
sens_A_EOQ = pd.DataFrame(
    index=["hA -20%", "hA +20%"],
    columns=["DA -20%", "DA +20%"],
    dtype=float
)

for lv_h in levels:
    for lv_D in levels:
        DA_test = DA * (1 + lv_D)
        hA_test = hA * (1 + lv_h)
        EOQ_test = math.sqrt(2 * DA_test * KA / hA_test)

        row = "hA -20%" if lv_h == -0.2 else "hA +20%"
        col = "DA -20%" if lv_D == -0.2 else "DA +20%"
        sens_A_EOQ.loc[row, col] = EOQ_test

# =========================
# SKU B – (Q,R) Model
# =========================

EOQ_B = math.sqrt(2 * DB_annual * KB / hB)          # EOQ

mu_LT = muB * L                                     # μέση ζήτηση στο lead time
sigma_LT = sigmaB * math.sqrt(L)                    # τυπ. απόκλιση στο lead time

zB = float(norm.ppf(CSL))                           # z για CSL
SS_B = zB * sigma_LT                                # safety stock
R_B = mu_LT + SS_B                                  # reorder point

ordering_cost_B = (DB_annual / EOQ_B) * KB          # ordering cost
holding_cost_B = (EOQ_B / 2 + SS_B) * hB            # holding cost (incl SS)
total_cost_B = ordering_cost_B + holding_cost_B

results_B_policy = pd.DataFrame([{
    "EOQ_B": EOQ_B,
    "mu_LT": mu_LT,
    "sigma_LT": sigma_LT,
    "CSL": CSL,
    "z(CSL)": zB,
    "SafetyStock_SS": SS_B,
    "ReorderPoint_R": R_B
}])

results_B_costs = pd.DataFrame([{
    "OrderingCost_B": ordering_cost_B,
    "HoldingCost_B(incl_SS)": holding_cost_B,
    "TotalAnnualCost_B": total_cost_B
}])

# =========================
# SKU C – Newsvendor
# =========================

Cu = pC - cC                                         # underorder cost
Co = cC - vC                                         # overorder cost
CR = Cu / (Cu + Co)                                  # critical ratio

zC = float(norm.ppf(CR))                             # z(CR)
Q_star = muC + zC * sigmaC                            # Q* = μ + zσ

results_C = pd.DataFrame([{
    "Cu(p-c)": Cu,
    "Co(c-v)": Co,
    "CriticalRatio": CR,
    "z(CR)": zC,
    "Q_star": Q_star
}])

# Sensitivity σε σ (0.8σ, 1.0σ, 1.2σ)
factors = [0.8, 1.0, 1.2]
sens_C = pd.DataFrame([
    {"sigma_factor": f, "sigma_used": sigmaC * f, "Q_star": muC + zC * (sigmaC * f)}
    for f in factors
])


print("✅ SKU A – EOQ (Αποτελέσματα)")
display(results_A)

print("\n✅ SKU A – Ευαισθησία EOQ (±20% σε DA και hA)")
display(sens_A_EOQ)

print("\n✅ SKU B – (Q,R) Πολιτική (EOQ, R, SS)")
display(results_B_policy)

print("\n✅ SKU B – Ετήσια Κόστη")
display(results_B_costs)

print("\n✅ SKU C – Newsvendor (Αποτελέσματα)")
display(results_C)

print("\n✅ SKU C – Ευαισθησία Q* ως προς σ")
display(sens_C)


✅ SKU A – EOQ (Αποτελέσματα)


,EOQ_A,Orders_per_year_A,Annual_cost_A(ordering+holding)
0,4381.78046,10.954451,5258.136552



✅ SKU A – Ευαισθησία EOQ (±20% σε DA και hA)


,DA -20%,DA +20%
hA -20%,4381.780460,5366.563146
hA +20%,3577.708764,4381.780460



✅ SKU B – (Q,R) Πολιτική (EOQ, R, SS)


...

,OrderingCost_B,HoldingCost_B(incl_SS),TotalAnnualCost_B
0,4242.640687,4736.096775,8978.737462



✅ SKU C – Newsvendor (Αποτελέσματα)


,Cu(p-c),Co(c-v),CriticalRatio,z(CR),Q_star
0,21,12,0.636364,0.348756,5288.257974



✅ SKU C – Ευαισθησία Q* ως προς σ


,sigma_factor,sigma_used,Q_star
0,0.8,1120.0,5190.606379
1,1.0,1400.0,5288.257974
2,1.2,1680.0,5385.909568
